# Compare task script outputs

Configure an existing script and a modified script, generate ONNX from both, run a selected task example, and compare NeuroGolf score metrics.

In [36]:
from pathlib import Path

# Inputs
TASK_NUM = 2
EXISTING_SCRIPT = Path("scripts/task002.py")
MODIFIED_SCRIPT = Path("scripts/task002_modify.py")

# Used when a build_model(source_path) style script needs an ONNX baseline.
SOURCE_ONNX = Path("solution/6334.25/task002.onnx")

# Example selector for quick testing.
# SPLIT must be one of: "train", "test", "arc-gen".
SPLIT = "test"
CASE_INDEX = 0

# Full verification is slower. Set True when needed.
RUN_FULL_VERIFY = False

COMPARE_DIR = Path("outputs/script_compare/task002")
EXISTING_ONNX = COMPARE_DIR / "existing_task002.onnx"
MODIFIED_ONNX = COMPARE_DIR / "modified_task002.onnx"

In [37]:
import copy
import importlib.util
import inspect
import json
import math
import os
import sys
import time

import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "neurogolf-2026").is_dir() and (path / "scripts").is_dir():
            return path
    raise FileNotFoundError("project root not found")


ROOT = find_project_root()
DATA_DIR = ROOT / "neurogolf-2026"
SCRIPTS_DIR = ROOT / "scripts"
COMPARE_DIR = ROOT / COMPARE_DIR
EXISTING_SCRIPT = ROOT / EXISTING_SCRIPT
MODIFIED_SCRIPT = ROOT / MODIFIED_SCRIPT
SOURCE_ONNX = ROOT / SOURCE_ONNX
EXISTING_ONNX = ROOT / EXISTING_ONNX
MODIFIED_ONNX = ROOT / MODIFIED_ONNX
PROFILE_DIR = ROOT / "profiles" / "script-compare" / f"task{TASK_NUM:03d}"

COMPARE_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SCRIPTS_DIR))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib-cache"))

from neurogolf_common import grid_to_tensor, tensor_to_grid, verify_onnx

print("ROOT", ROOT)
print("existing", EXISTING_SCRIPT.relative_to(ROOT))
print("modified", MODIFIED_SCRIPT.relative_to(ROOT))
print("source onnx", SOURCE_ONNX.relative_to(ROOT))

ROOT /Users/kaiikeda/Programming/Kaggle/Neuro Golf
existing scripts/task002.py
modified scripts/task002_modify.py
source onnx solution/6334.25/task002.onnx


## Generate ONNX from scripts

In [38]:
def load_module(script_path: Path, name: str):
    spec = importlib.util.spec_from_file_location(name, script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def build_model_from_script(script_path: Path, output_path: Path, label: str):
    module = load_module(script_path, f"compare_{label}_{int(time.time() * 1000)}")
    if not hasattr(module, "build_model"):
        raise AttributeError(f"{script_path} does not define build_model()")

    signature = inspect.signature(module.build_model)
    required = [
        p for p in signature.parameters.values()
        if p.default is inspect.Signature.empty
        and p.kind in (p.POSITIONAL_ONLY, p.POSITIONAL_OR_KEYWORD)
    ]

    if len(required) == 0:
        model = module.build_model()
    elif len(required) == 1:
        model = module.build_model(SOURCE_ONNX)
    else:
        raise TypeError(f"unsupported build_model signature: {signature}")

    onnx.checker.check_model(model, full_check=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    onnx.save(model, output_path)
    return model


existing_model = build_model_from_script(EXISTING_SCRIPT, EXISTING_ONNX, "existing")
modified_model = build_model_from_script(MODIFIED_SCRIPT, MODIFIED_ONNX, "modified")

print("saved existing", EXISTING_ONNX.relative_to(ROOT), EXISTING_ONNX.stat().st_size)
print("saved modified", MODIFIED_ONNX.relative_to(ROOT), MODIFIED_ONNX.stat().st_size)

saved existing outputs/script_compare/task002/existing_task002.onnx 4352
saved modified outputs/script_compare/task002/modified_task002.onnx 4500


## Model summary

In [39]:
def summarize_model(path: Path):
    model = onnx.load(path)
    op_counts = {}
    for node in model.graph.node:
        op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1
    return {
        "label": path.stem,
        "path": str(path.relative_to(ROOT)),
        "bytes": path.stat().st_size,
        "ir_version": model.ir_version,
        "opsets": [(op.domain, op.version) for op in model.opset_import],
        "nodes": len(model.graph.node),
        "initializers": len(model.graph.initializer),
        "value_info": len(model.graph.value_info),
        "ops": dict(sorted(op_counts.items())),
    }


pd.DataFrame([summarize_model(EXISTING_ONNX), summarize_model(MODIFIED_ONNX)])

,label,path,bytes,ir_version,opsets,nodes,initializers,value_info,ops
0,existing_task002,outputs/script_compare/task002/existing_task00...,4352,10,"[(, 17)]",86,10,0,"{'And': 1, 'Cast': 3, 'Concat': 1, 'Conv': 24,..."
1,modified_task002,outputs/script_compare/task002/modified_task00...,4500,10,"[(, 17)]",72,10,0,"{'Cast': 2, 'Concat': 1, 'Conv': 20, 'Greater'..."


## Selected example test

In [40]:
def load_task(task_num):
    with (DATA_DIR / f"task{task_num:03d}.json").open() as f:
        return json.load(f)


def run_case(path: Path, example):
    session = ort.InferenceSession(path.read_bytes())
    output = session.run(["output"], {"input": grid_to_tensor(example["input"])})[0]
    return tensor_to_grid(output)


task = load_task(TASK_NUM)
example = task[SPLIT][CASE_INDEX]
existing_actual = run_case(EXISTING_ONNX, example)
modified_actual = run_case(MODIFIED_ONNX, example)
expected = example["output"]

case_result = pd.DataFrame([
    {"label": "existing", "ok": existing_actual == expected, "actual": existing_actual},
    {"label": "modified", "ok": modified_actual == expected, "actual": modified_actual},
])
print(f"task{TASK_NUM:03d} {SPLIT}[{CASE_INDEX}]")
print("input:", example["input"])
print("expected:", expected)
case_result

task002 test[0]
input: [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 3, 3, 3, 3, 3, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3, 0, 0, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0, 3, 3, 0, 3, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0], [0, 0, 3, 0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 3, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 

,label,ok,actual
0,existing,True,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
1,modified,True,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."


## Optional full verification

In [41]:
if RUN_FULL_VERIFY:
    full_verify = pd.DataFrame([
        {"label": "existing", "passed": verify_onnx(EXISTING_ONNX, task)},
        {"label": "modified", "passed": verify_onnx(MODIFIED_ONNX, task)},
    ])
else:
    full_verify = pd.DataFrame([{"label": "skipped", "passed": None}])
full_verify

,label,passed
0,skipped,None


## Score comparison

In [42]:
def load_neurogolf_utils():
    spec = importlib.util.spec_from_file_location(
        "neurogolf_utils",
        DATA_DIR / "neurogolf_utils" / "neurogolf_utils.py",
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module._NEUROGOLF_DIR = str(DATA_DIR) + "/"
    return module


def score_path(path: Path, label: str, profile_example):
    ng = load_neurogolf_utils()
    model = onnx.load(path)
    sanitized = ng.sanitize_model(copy.deepcopy(model))
    if sanitized is None:
        raise ValueError(f"sanitize_model failed for {path}")

    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    options.profile_file_prefix = str(PROFILE_DIR / label)
    session = ort.InferenceSession(sanitized.SerializeToString(), options)
    session.run(["output"], {"input": grid_to_tensor(profile_example["input"])})
    profile_path = session.end_profiling()
    memory, params = ng.score_network(sanitized, profile_path)
    score = None
    if memory is not None and params is not None:
        score = max(1.0, 25.0 - math.log(max(1.0, memory + params)))
    return {
        "label": label,
        "path": str(path.relative_to(ROOT)),
        "bytes": path.stat().st_size,
        "memory": memory,
        "params": params,
        "score": score,
        "profile": str(Path(profile_path).relative_to(ROOT)),
    }


score_df = pd.DataFrame([
    score_path(EXISTING_ONNX, "existing", example),
    score_path(MODIFIED_ONNX, "modified", example),
])
score_df["score_delta_vs_existing"] = score_df["score"] - score_df.loc[score_df["label"] == "existing", "score"].iloc[0]
score_df

,label,path,bytes,memory,params,score,profile,score_delta_vs_existing
0,existing,outputs/script_compare/task002/existing_task00...,4352,97200,37,13.515093,profiles/script-compare/task002/existing_2026-...,0.000000
1,modified,outputs/script_compare/task002/modified_task00...,4500,87200,436,13.619053,profiles/script-compare/task002/modified_2026-...,0.103959
